Mini Project — Part A
Evolutionary AI

A1 — Genome Representation


In [ ]:
import random

def random_individual(length: int, rng) -> list[int]:
    random.seed(rng)
    assert isinstance(length, int)
    assert length > 0

    steps = [random.randint(0, 3) for _ in range(length)]
    return steps


Testing A1

In [ ]:
rng = 5
examples = random_individual(4, rng)
print(examples)


With fixed length encoding, all genomes have the same size. This makes crossover and mutation easier, because children always have the same length as their parents. This way, paths can evolve without problems related to different genome sizes.


A2 — Mutation Operator

In [ ]:
def mutate_path(path, p, rng) -> list[int]:
    random.seed(rng)
    new_path = path.copy()

    for i in range(len(new_path)):
        if random.random() < p:
            new_path[i] = random.randint(0, 3)

    return new_path


Testing A2

In [ ]:
rng = 100
first_path = [0, 0, 0, 0]

print(f"The original path is: {first_path}")

p01 = mutate_path(first_path, p=0.1, rng=rng)
print(f"Path where p=0.1: {p01}")

p0 = mutate_path(first_path, p=0.0, rng=rng)
print(f"Path where p=0.0: {p0}")

p1 = mutate_path(first_path, p=1.0, rng=rng)
print(f"Path where p=1.0: {p1}")

With a high mutation probability, exploration is high because many genes change. With a low mutation probability, most genes stay the same, which means less exploration and more stability.


A3 — Crossover Operator

In [ ]:
def crossover_path(p1: list[int], p2: list[int], rng):
    random.seed(rng)

    if len(p1) != len(p2):
        raise ValueError("Parents must have the same length")

    length = len(p1)
    crossover_point = random.randint(1, length - 1)

    child1 = p1[:crossover_point] + p2[crossover_point:]
    child2 = p2[:crossover_point] + p1[crossover_point:]

    return child1, child2

Testing A3

In [ ]:
parenta = [0,0,0,0,0]
parentb = [1,1,1,1,1]

c1, c2 = crossover_path(parenta, parentb, rng=42)

print("Parent A:", parenta)
print("Parent B:", parentb)
print("Child 1:", c1)
print("Child 2:", c2)

A4 — Tournament Selection

In [ ]:
def tournament_select(population, k, fit_func, rng):
    random.seed(rng)

    k = min(k, len(population))
    contestants = random.sample(population, k)

    best = contestants[0]
    best_fitness = fit_func(best)

    for ind in contestants:
        f = fit_func(ind)
        if f > best_fitness:
            best_fitness = f
            best = ind

    return best


Testing A4

In [ ]:
population = [1, 2, 3, 4, 5]

def getting_score(ind):
    return ind

print("k = 1")
for i in range(5):
    print(tournament_select(population, 1, getting_score, rng=40+i))

print("k = 5")
for i in range(5):
    print(tournament_select(population, 5, getting_score, rng=40+i))


With a low k, weaker individuals can still win, which keeps diversity high. With a high k, the strongest individuals almost always win, which increases selection pressure but reduces diversity.


A5 — Environment Simulation

In [ ]:
def simulate_path(genome: list[int], grid):
    x, y = 0, 0
    collisions = 0

    moves = {
        0: (0, -1),
        1: (0, 1),
        2: (-1, 0),
        3: (1, 0)
    }

    for move in genome:
        dx, dy = moves[move]
        nx, ny = x + dx, y + dy

        if nx < 0 or ny < 0 or ny >= len(grid) or nx >= len(grid[0]):
            collisions += 1
        elif grid[ny][nx] == 1:
            collisions += 1
        else:
            x, y = nx, ny

    return (x, y), collisions


Testing A5

In [ ]:
grid = [
    [0, 0, 1],
    [0, 1, 0],
    [0, 0, 0]
]

print(simulate_path([1,1,3,3], grid))
print(simulate_path([1,3,1,3], grid))


A6 — Fitness Function

In [ ]:
def fitness(genome, grid, goal=(9,9)):
    (x, y), c = simulate_path(genome, grid)
    xg, yg = goal

    d = abs(x - xg) + abs(y - yg)

    if d == 0 and c == 0:
        return 1.0

    return 1 / (d + c)


The fitness function rewards paths that end closer to the goal and have fewer collisions. Distance and collisions have the same influence because both are added in the denominator.


A7 — Conceptual Reflection

Variation and selection lead to improvement because many different paths are created and tested. Some paths perform better by ending closer to the goal or having fewer collisions. Selection keeps these better paths more often, so good parts of paths appear more frequently over time.

In A1 the path representation defines which moves are possible. A2 introduces changes through mutation, which allows exploration. A3 combines paths, which can merge good parts of different solutions. A4 selects better paths more often, so useful genes survive. Together, these steps allow the algorithm to improve paths without knowing the correct solution.
